# Constructor GraphRAG Industrial con Neo4j

Indexa documentos como `Document -> Chunk -> Entity`, extrae relaciones con LLM
y deja Neo4j listo para retrieval GraphRAG con embeddings, provenance e índices.

El grafo resultante puede ser consultado por el orquestador vía `graph_query_entity`,
`graph_get_schema`, `execute_cypher`, etc.

## Cómo usar

1. Ejecuta celdas en orden
2. Configura el **provider** (OpenAI o Ollama)
3. Define los **tipos de nodos y relaciones permitidos**
4. Pega tu texto o carga archivos
5. Ejecuta la extracción → se crean entidades y relaciones en Neo4j
6. Explora el grafo resultante

---
## 1. Configuración

In [ ]:
import os
import re
import json
import hashlib
from pathlib import Path
from typing import Optional

from dotenv import load_dotenv
from neo4j import GraphDatabase, Driver
from pydantic import BaseModel, Field, AliasChoices
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate

load_dotenv("../.env")
print("Importaciones ok")

In [ ]:
# ============================================================
# ELIGE PROVIDER: "openai" o "ollama"
# ============================================================
LLM_PROVIDER = "openai"

OLLAMA_EXTRACT_MODEL = "lfm2:24b"        # usado si provider = "ollama"
OPENAI_EXTRACT_MODEL = "gpt-4o-mini"      # usado si provider = "openai"

# Embeddings siempre vía Ollama (qwen3-embedding:latest)
EMBEDDINGS_MODEL = os.getenv("EMBEDDINGS_MODEL", "qwen3-embedding:latest")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

print(f"Provider: {LLM_PROVIDER}")
print(f"Modelo extracción: {OLLAMA_EXTRACT_MODEL if LLM_PROVIDER == 'ollama' else OPENAI_EXTRACT_MODEL}")
print(f"Modelo embeddings: {EMBEDDINGS_MODEL}")

In [ ]:
# Conexión a Neo4j (usando el mismo patrón que graph_tools)
_driver: Optional[Driver] = None


def ensure_driver() -> Driver:
    global _driver
    if _driver is not None:
        return _driver

    uri = os.getenv("NEO4J_URI", "bolt://localhost:7687")
    user = os.getenv("NEO4J_USER", "neo4j")
    password = os.getenv("NEO4J_PASSWORD", "deepagents")

    _driver = GraphDatabase.driver(uri, auth=(user, password))
    _driver.verify_connectivity()
    print(f"Conectado a Neo4j en {uri}")
    return _driver


def run_query(query: str, params: Optional[dict] = None) -> list:
    with ensure_driver().session() as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]


def run_write(query: str, params: Optional[dict] = None) -> None:
    with ensure_driver().session() as session:
        session.execute_write(lambda tx: list(tx.run(query, params or {})))


def ensure_constraints() -> None:
    """Crea constraints e índices base (no necesita embeddings)."""
    statements = [
        "CREATE CONSTRAINT document_id IF NOT EXISTS FOR (d:Document) REQUIRE d.id IS UNIQUE",
        "CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (c:Chunk) REQUIRE c.id IS UNIQUE",
        "CREATE CONSTRAINT entity_id IF NOT EXISTS FOR (e:Entity) REQUIRE e.id IS UNIQUE",
        "CREATE INDEX entity_name IF NOT EXISTS FOR (e:Entity) ON (e.name)",
        "CREATE INDEX entity_type IF NOT EXISTS FOR (e:Entity) ON (e.type)",
        "CREATE FULLTEXT INDEX entity_text IF NOT EXISTS FOR (e:Entity) ON EACH [e.name, e.description]",
    ]
    for statement in statements:
        run_write(statement)
    print("Constraints y fulltext index creados")


ensure_driver()
ensure_constraints()

In [ ]:
# Embeddings para GraphRAG (opcional)
# Si Ollama no responde, embeddings queda como None y el gráfico funciona sin vectores.
try:
    embeddings = OllamaEmbeddings(
        model=EMBEDDINGS_MODEL,
        base_url=OLLAMA_BASE_URL,
    )
    # Probe rápida para verificar que responde
    _ = len(embeddings.embed_query("ping"))
    print("Embeddings cargados y respondiendo.")
except Exception as e:
    print(f"⚠️  Embeddings no disponibles: {e}")
    embeddings = None

In [ ]:
# ============================================================
# CREAR VECTOR INDEX (requiere Ollama respondiendo)
# ============================================================
# Si Ollama no está corriendo, falla suave sin bloquear el notebook.
# El retrieval caerá a búsqueda fulltext. Vuelve a ejecutar tras iniciar Ollama.

existing = run_query(
    "SHOW VECTOR INDEXES WHERE name = 'chunk_embedding'"
)
if existing:
    dims = existing[0].get("dimensions", None)
    print(f"Vector index chunk_embedding ya existe ({dims} dims)")
elif embeddings is None:
    print("⚠️  embeddings no disponible (celda anterior falló). No se puede crear vector index.")
else:
    try:
        dims = len(embeddings.embed_query("probe"))
        run_write(f"""
        CREATE VECTOR INDEX chunk_embedding IF NOT EXISTS
        FOR (c:Chunk) ON (c.embedding)
        OPTIONS {{indexConfig: {{
            `vector.dimensions`: {dims},
            `vector.similarity_function`: 'cosine'
        }}}}
        """)
        print(f"Vector index chunk_embedding creado ({dims} dims)")
    except Exception as e:
        print(f"⚠️  Vector index no creado (Ollama no responde?): {e}")

---
## 2. Esquema del grafo

Define qué tipos de entidades y relaciones puede extraer el LLM.
Edita estas listas antes de ejecutar la extracción.

In [ ]:
# ============================================================
# DEFINE AQUÍ LOS TIPOS DE NODOS Y RELACIONES PERMITIDOS
# ============================================================
ALLOWED_NODES = [
    "Concepto",
    "Framework",
    "Componente",
    "Clase",
    "Herramienta",
    "Workflow",
    "Prompt",
    "Modelo",
    "Biblioteca",
    "Patron",
]

ALLOWED_RELATIONSHIPS = [
    "CONECTA_CON",
    "HEREDA_DE",
    "UTILIZA",
    "CONFIGURA",
    "IMPLEMENTA",
    "EJECUTA",
    "DEPENDE_DE",
    "GENERA",
    "ALIMENTA",
    "COMPONE",
]

print(f"{len(ALLOWED_NODES)} tipos de nodo permitidos")
print(f"{len(ALLOWED_RELATIONSHIPS)} tipos de relación permitidos")

---
## 3. Modelos de extracción (Pydantic)

In [ ]:
class Entity(BaseModel):
    """Entidad extraída del texto."""
    entidad: str = Field(default="", validation_alias=AliasChoices("entidad", "nombre"), description="Nombre de la entidad")
    tipo: str = Field(default="", description=f"Tipo. Debe ser uno de: {', '.join(ALLOWED_NODES)}")
    descripcion: str = Field(default="", validation_alias=AliasChoices("descripcion", "description"), description="Descripción")


class Relationship(BaseModel):
    """Relación entre dos entidades."""
    sujeto: str = Field(default="", validation_alias=AliasChoices("sujeto", "source"), description="Entidad origen")
    relacion: str = Field(default="", validation_alias=AliasChoices("relacion", "rel_type"), description=f"Tipo. Debe ser uno de: {', '.join(ALLOWED_RELATIONSHIPS)}")
    objeto: str = Field(default="", validation_alias=AliasChoices("objeto", "target"), description="Entidad destino")


class GraphExtraction(BaseModel):
    """Resultado completo de la extracción."""
    entidades: list[Entity] = Field(validation_alias=AliasChoices("entidades", "entities"), default_factory=list)
    relaciones: list[Relationship] = Field(validation_alias=AliasChoices("relaciones", "relationships"), default_factory=list)


print("Modelos Pydantic definidos")

---
## 4. Extractor LLM

Crea el extractor a partir del provider elegido y las listas de nodos/relaciones.

In [ ]:
def build_extractor(
    provider: str,
    allowed_nodes: list[str],
    allowed_relationships: list[str],
    method_override: Optional[str] = None,
) -> object:
    """Construye un extractor con structured output.
    Retorna un callable que acepta {"text": ...} y devuelve GraphExtraction."""

    system_prompt = f"""Eres un extractor de grafos de conocimiento.

Extrae del chunk proporcionado las entidades y relaciones relevantes para GraphRAG.

REGLAS:
- Las entidades SOLO pueden ser de estos tipos: {', '.join(allowed_nodes)}.
- Las relaciones SOLO pueden ser de estos tipos: {', '.join(allowed_relationships)}.
- Cada entidad debe tener un nombre canónico, estable y específico. Evita pronombres, nombres genéricos y duplicados triviales.
- Cada relación conecta dos entidades extraídas en este mismo chunk por su nombre canónico.
- Si el texto menciona una relación no cubierta por la lista, elige la más cercana o no la incluyas.
- Prioriza precisión sobre cantidad: extrae entidades útiles para búsqueda y razonamiento, no cada palabra clave.
- Si no hay suficientes datos para una entidad, no la incluyas.
- Responde únicamente en formato JSON estructurado.
"""

    if provider == "openai":
        llm = ChatOpenAI(model=OPENAI_EXTRACT_MODEL, temperature=0)
        method = method_override or "function_calling"
    else:
        llm = ChatOllama(model=OLLAMA_EXTRACT_MODEL, temperature=0)
        method = method_override or "json_mode"

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{text}"),
    ])

    return prompt | llm.with_structured_output(GraphExtraction, method=method)


extractor = build_extractor(LLM_PROVIDER, ALLOWED_NODES, ALLOWED_RELATIONSHIPS)
print(f"Extractor creado con {LLM_PROVIDER}")

In [ ]:
def canonical_id(value: str) -> str:
    """ID estable para deduplicar entidades entre chunks."""
    normalized = re.sub(r"\s+", " ", value.strip().lower())
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()[:16]


def document_id_from_file(path: Path) -> str:
    # deepagents_mainpy_chunk_1.md -> deepagents_mainpy
    stem = re.sub(r"_chunk_\d+$", "", path.stem)
    return stem or path.stem


def insert_graph_extraction(
    extraction: GraphExtraction,
    source_text: str = "",
    source_name: str = "manual",
    chunk_id: Optional[str] = None,
    document_id: Optional[str] = None,
    chunk_index: Optional[int] = None,
    store_embedding: bool = True,
):
    """Inserta el grafo con patrón industrial GraphRAG.

    Modelo: (:Document)-[:HAS_CHUNK]->(:Chunk)-[:MENTIONS]->(:Entity).
    Las relaciones semánticas entre entidades quedan en :REL con type,
    source_chunk_id y source_document_id para trazabilidad.
    """
    created_entities = 0
    created_rels = 0

    text_hash = hashlib.sha256(source_text.encode("utf-8")).hexdigest()[:16] if source_text else "empty"
    doc_id = document_id or source_name or "manual"
    cid = chunk_id or f"{doc_id}:{text_hash}"
    chunk_embedding = embeddings.embed_query(source_text[:4000]) if store_embedding and source_text and embeddings is not None else None

    run_write(
        """
        MERGE (d:Document {id: $doc_id})
        SET d.name = $source_name,
            d.updated_at = datetime()
        MERGE (c:Chunk {id: $chunk_id})
        SET c.text = $text,
            c.name = $source_name,
            c.content_hash = $content_hash,
            c.chunk_index = $chunk_index,
            c.embedding = $embedding,
            c.updated_at = datetime()
        MERGE (d)-[:HAS_CHUNK]->(c)
        """,
        {
            "doc_id": doc_id,
            "source_name": source_name,
            "chunk_id": cid,
            "text": source_text,
            "content_hash": text_hash,
            "chunk_index": chunk_index,
            "embedding": chunk_embedding,
        },
    )

    entity_names = set()
    for entity in extraction.entidades:
        name = entity.entidad.strip()
        if not name or entity.tipo not in ALLOWED_NODES or entity.tipo == "Chunk":
            continue

        entity_names.add(name)
        eid = canonical_id(name)
        run_write(
            """
            MATCH (c:Chunk {id: $chunk_id})
            MERGE (e:Entity {id: $entity_id})
            SET e.name = $name,
                e.type = $type,
                e.description = CASE WHEN $description <> "" THEN $description ELSE coalesce(e.description, "") END,
                e.updated_at = datetime()
            MERGE (c)-[m:MENTIONS]->(e)
            SET m.source_document_id = $doc_id,
                m.updated_at = datetime()
            """,
            {
                "chunk_id": cid,
                "entity_id": eid,
                "name": name,
                "type": entity.tipo,
                "description": (entity.descripcion or "").strip(),
                "doc_id": doc_id,
            },
        )
        created_entities += 1

    for rel in extraction.relaciones:
        source = rel.sujeto.strip()
        target = rel.objeto.strip()
        rel_type = rel.relacion.strip().upper()
        if not source or not target or source == target or rel_type not in ALLOWED_RELATIONSHIPS:
            continue
        if source not in entity_names or target not in entity_names:
            continue

        run_write(
            """
            MATCH (a:Entity {id: $source_id})
            MATCH (b:Entity {id: $target_id})
            MERGE (a)-[r:REL {type: $rel_type}]->(b)
            SET r.source_chunk_ids = apoc.coll.toSet(coalesce(r.source_chunk_ids, []) + [$chunk_id]),
                r.source_document_ids = apoc.coll.toSet(coalesce(r.source_document_ids, []) + [$doc_id]),
                r.updated_at = datetime()
            """,
            {
                "source_id": canonical_id(source),
                "target_id": canonical_id(target),
                "rel_type": rel_type,
                "chunk_id": cid,
                "doc_id": doc_id,
            },
        )
        created_rels += 1

    return created_entities, created_rels


---
## 5. Texto de entrada

Pega tu texto aquí o carga archivos.

In [ ]:
# ============================================================
# OPCIÓN A: Pega tu texto directamente aquí
# ============================================================
SOURCE_TEXT = """
LangChain es un framework para construir aplicaciones con LLMs.
LangGraph es una extensión de LangChain para crear agentes y flujos de trabajo.
ChromaDB es una base de datos vectorial utilizada para almacenar embeddings.
Ollama permite ejecutar modelos de lenguaje localmente, como Llama 3.1 y Qwen.
Neo4j es un grafo de conocimiento que almacena entidades y relaciones.

El orquestador utiliza subagentes especializados para indexar y recuperar información.
Los embeddings se generan con el modelo qwen3-embedding a través de Ollama.
El agente python_indexer indexa fragmentos de código en ChromaDB.
El agente python_retriever busca conocimiento almacenado en el vector store.
Todo el sistema se despliega con Docker Desktop y LangGraph API Server.
"""

print(f"Texto cargado: {len(SOURCE_TEXT)} caracteres")

In [ ]:
# ============================================================
# OPCIÓN B: Cargar desde archivos
# ============================================================
# Descomenta y edita la ruta para cargar archivos en vez del texto de arriba:

# file_paths = [
#     "../data/chunks/mi_archivo.md",
#     "../data/chunks/otro_archivo.txt",
# ]
#
# combined = []
# for fp in file_paths:
#     p = Path(fp)
#     if p.exists():
#         combined.append(f"--- {p.name} ---\n{p.read_text(encoding='utf-8')}")
# SOURCE_TEXT = "\n\n".join(combined)
# print(f"Cargados {len(file_paths)} archivos, {len(SOURCE_TEXT)} caracteres totales")

In [ ]:
# ============================================================
# OPCIÓN C: Procesar solo archivos NUEVOS en data/chunks/
# ============================================================
chunks_dir = Path("../data/chunks")

if chunks_dir.exists():
    print("=== INICIANDO INDEXACIÓN INTELIGENTE (SOLO ARCHIVOS NUEVOS) ===")
    
    # Obtenemos la lista de todos los archivos en la carpeta ordenados
    files = sorted([f for f in chunks_dir.iterdir() if f.is_file()])
    
    total_entities_global = 0
    total_rels_global = 0
    skipped_files = 0

    for f in files:
        chunk_content = f.read_text(encoding='utf-8')
        doc_id = document_id_from_file(f)
        match = re.search(r"_chunk_(\d+)", f.stem)
        chunk_index = int(match.group(1)) if match else None
        chunk_hash = hashlib.sha256(chunk_content.encode("utf-8")).hexdigest()[:16]
        chunk_id = f"{doc_id}:{f.name}:{chunk_hash}"

        # VERIFICACIÓN: idempotencia real por documento + archivo + hash de contenido.
        check_query = """
        MATCH (c:Chunk {id: $chunk_id})
        RETURN count(c) AS count
        """
        exists_result = run_query(check_query, {"chunk_id": chunk_id})
        
        if exists_result and exists_result[0]["count"] > 0:
            # Si el archivo ya está registrado, lo saltamos de inmediato
            skipped_files += 1
            continue

        print(f"\n⚡ [NUEVO] Procesando archivo: {f.name}...")
        
        # 1. Ejecutar el LLM extractor para este fragmento específico
        # Con fallback automático: si el método principal falla, reintenta con el alternativo
        result: Optional[GraphExtraction] = None
        fallback_method = "json_mode" if LLM_PROVIDER == "openai" else "function_calling"
        for attempt, current_method in enumerate([None, fallback_method]):
            try:
                if attempt == 0:
                    result = extractor.invoke({"text": chunk_content})  # type: ignore
                else:
                    print(f"   ↻ Reintentando con método '{current_method}'...")
                    fallback_extractor = build_extractor(LLM_PROVIDER, ALLOWED_NODES, ALLOWED_RELATIONSHIPS, method_override=current_method)
                    result = fallback_extractor.invoke({"text": chunk_content})  # type: ignore
                break
            except Exception as e:
                if attempt == 0:
                    print(f"⚠️  Error con método principal: {e}")
                else:
                    print(f"❌ Error también con método alternativo {current_method}: {e}")
                    result = None

        if result is None:
            print(f"❌ No se pudo procesar el archivo {f.name} con ningún método.")
            continue

        print(f"   -> Extraídas {len(result.entidades)} entidades y {len(result.relaciones)} relaciones.")

        # 2. Inserción GraphRAG: Document -> Chunk -> Entity, con provenance y embedding en Chunk
        entities_count, rels_count = insert_graph_extraction(
            result,
            source_text=chunk_content,
            source_name=f.name,
            chunk_id=chunk_id,
            document_id=doc_id,
            chunk_index=chunk_index,
            store_embedding=True,
        )
        total_entities_global += entities_count
        total_rels_global += rels_count


    run_write("""
    MATCH (d:Document)-[:HAS_CHUNK]->(c:Chunk)
    WHERE c.chunk_index IS NOT NULL
    WITH d, c ORDER BY d.id, c.chunk_index
    WITH d, collect(c) AS chunks
    WHERE size(chunks) > 1
    UNWIND range(0, size(chunks) - 2) AS i
    WITH chunks[i] AS c1, chunks[i + 1] AS c2
    MERGE (c1)-[:NEXT]->(c2)
    """)

    print("\n====================================================")
    print("🎉 ¡PROCESO TERMINADO!")
    print(f"Archivos ya existentes omitidos: {skipped_files}")
    print(f"Nuevos datos indexados: {total_entities_global} entidades y {total_rels_global} relaciones.")
else:
    print("❌ Error: La ruta ../data/chunks/ no existe localmente para este notebook.")

---
## 5d. Post-linkeo entre chunks

Conecta entidades que fueron extraídas en distintos chunks pero que están
relacionadas por co-ocurrencia o porque pertenecen al mismo documento.

> **CO_OCURRE_EN**: entidades mencionadas juntas en un mismo chunk.
> **MISMO_DOCUMENTO**: entidades que aparecen en diferentes chunks del mismo archivo fuente.

Ejecuta ambas celdas **después** de procesar los chunks (Opción C).

In [ ]:
# ============================================================
# POST-LINKEO BÁSICO: entidades que coexisten en el mismo chunk
# ============================================================
print("🔗 Post-linkeo: conectando entidades que coexisten en el mismo chunk...")

# Batch por documento para evitar O(N²) en memoria
docs = run_query("MATCH (d:Document) RETURN d.id AS id")
for doc in docs:
    run_query("""
        MATCH (d:Document {id: $doc_id})-[:HAS_CHUNK]->(c:Chunk)-[:MENTIONS]->(e1:Entity)
        MATCH (c)-[:MENTIONS]->(e2:Entity)
        WHERE e1.id < e2.id
        WITH e1, e2, count(DISTINCT c) AS veces
        MERGE (e1)-[r:CO_OCURRE_EN]->(e2)
        SET r.weight = veces
    """, {"doc_id": doc["id"]})

counts = run_query(
    "MATCH ()-[r:CO_OCURRE_EN]->() RETURN count(r) AS total"
)
total = counts[0]["total"] if counts else 0
print(f"✅ Creadas {total} relaciones CO_OCURRE_EN entre entidades.")


In [ ]:
# ============================================================
# POST-LINKEO AVANZADO: entidades en chunks del mismo archivo
# ============================================================
# Conecta entidades que aparecen en diferentes chunks pero
# cuyo nombre de chunk comparte el mismo prefijo (mismo archivo fuente).
# Batch por documento para evitar O(N²) en memoria.

print("🔗 Conectando entidades que aparecen en chunks del mismo documento...")

docs = run_query("MATCH (d:Document) RETURN d.id AS id")
for doc in docs:
    run_query("""
        MATCH (d:Document {id: $doc_id})-[:HAS_CHUNK]->(c1:Chunk)-[:MENTIONS]->(e1:Entity)
        MATCH (d)-[:HAS_CHUNK]->(c2:Chunk)-[:MENTIONS]->(e2:Entity)
        WHERE e1.id < e2.id
          AND c1 <> c2
        WITH e1, e2, count(DISTINCT c1) AS veces
        MERGE (e1)-[r:MISMO_DOCUMENTO]->(e2)
        SET r.weight = veces
    """, {"doc_id": doc["id"]})

counts = run_query(
    "MATCH ()-[r:MISMO_DOCUMENTO]->() RETURN count(r) AS total"
)
total = counts[0]["total"] if counts else 0
print(f"✅ Creadas {total} relaciones MISMO_DOCUMENTO entre entidades.")


---
## 6. Pipeline: Extracción → Inserción en Neo4j

Ejecuta el LLM, toma el resultado estructurado y lo inserta en Neo4j.

In [ ]:
print("Extrayendo entidades y relaciones con LLM...")
result: GraphExtraction = extractor.invoke({"text": SOURCE_TEXT})  # type: ignore

print(f"\nExtraídas {len(result.entidades)} entidades y {len(result.relaciones)} relaciones")

if result.entidades:
    print("\n--- ENTIDADES ---")
    for e in result.entidades:
        print(f"  [{e.tipo}] {e.entidad}: {e.descripcion[:80] if e.descripcion else e.entidad}...")

if result.relaciones:
    print("\n--- RELACIONES ---")
    for r in result.relaciones:
        print(f"  {r.sujeto} --[{r.relacion}]--> {r.objeto}")

In [ ]:


entities_count, rels_count = insert_graph_extraction(result, source_text=SOURCE_TEXT, store_embedding=True)
print(f"Insertadas {entities_count} entidades y {rels_count} relaciones en Neo4j")

---
## 6b. Resetear el grafo (opcional)

Ejecuta esta celda para borrar TODAS las entidades y relaciones del grafo Neo4j.
Útil para empezar desde cero sin reiniciar el contenedor.


In [ ]:
# BORRA TODAS LAS ENTIDADES Y RELACIONES DEL GRAFO
# Seguridad industrial: por defecto NO borra nada. Cambia a True manualmente.

CONFIRM_RESET = False
if CONFIRM_RESET:
    run_write("MATCH (n) DETACH DELETE n")
    print("Grafo limpiado: todas las entidades y relaciones eliminadas")
else:
    print("Reset no ejecutado. Cambia CONFIRM_RESET = True para borrar el grafo.")


---
## 7. Explorar el grafo resultante

In [ ]:
# Esquema actual del grafo
labels = run_query("CALL db.labels() YIELD label RETURN label ORDER BY label")
rel_types = run_query("CALL db.relationshipTypes() YIELD relationshipType AS rel RETURN rel ORDER BY rel")

print("=== ESQUEMA DEL GRAFO NEO4J ===")
print(f"\nLabels de nodos ({len(labels)}):")
for l in labels:
    print(f"  - {l['label']}")

print(f"\nTipos de relación ({len(rel_types)}):")
for r in rel_types:
    print(f"  - :{r['rel']}")

In [ ]:
# Listar entidades con sus relaciones
entities = run_query(
    """
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-(related:Entity)
    WITH n, r, related
    ORDER BY n.name
    RETURN n.name AS entity, n.type AS type, n.description AS description,
           collect(DISTINCT {
               relation: coalesce(type(r), ''),
               related_entity: coalesce(related.name, '')
           }) AS relationships
    """
)

for e in entities:
    rels = [r for r in e["relationships"] if r["related_entity"]]
    print(f"[{e['type']}] {e['entity']}")
    print(f"   Nota/Descripción: {e['description']}")
    for r in rels:
        print(f"  └── {r['relation']} → {r['related_entity']}")

In [ ]:
counts = run_query(
    """
    MATCH (n:Entity)
    WITH count(n) AS total_entities
    MATCH ()-[r]->()
    RETURN total_entities, count(r) AS total_relationships
    """
)

if counts:
    c = counts[0]
    print(f"Total entidades: {c['total_entities']}")
    print(f"Total relaciones: {c['total_relationships']}")

In [ ]:
# ============================================================
# POST-LINKEO: explorar conexiones entre chunks
# ============================================================
# Muestra las entidades con mayor co-ocurrencia entre chunks

cooc = run_query("""
    MATCH (e1:Entity)-[r:CO_OCURRE_EN]->(e2:Entity)
    RETURN e1.name AS origen, e2.name AS destino, r.weight AS peso
    ORDER BY r.weight DESC
    LIMIT 15
""")

if cooc:
    print("\n🔗 Top 15 conexiones CO_OCURRE_EN:")
    for c in cooc:
        bar = "█" * min(c["peso"], 20)
        print(f"   {c['origen']} ↔ {c['destino']}  (peso: {c['peso']}) {bar}")
else:
    print("\n(No hay relaciones CO_OCURRE_EN. Ejecuta primero la celda de post-linkeo 5d.)")

---
## 8. (Opcional) GraphRAG Query

Haz una pregunta y obtén contexto del grafo combinando
búsqueda vectorial + relaciones.

In [ ]:
def graphrag_query(question: str, top_k: int = 5, min_score: float = 0.25):
    """Retrieval GraphRAG: vector en Chunk + expansión local del grafo."""
    if embeddings is None:
        print("Embeddings no disponibles. Ejecuta la celda de embeddings con Ollama activo.")
        return []
    q_emb = embeddings.embed_query(question)

    results = run_query(
        """
        MATCH (c:Chunk)
        WHERE c.embedding IS NOT NULL
        WITH c, vector.similarity.cosine(c.embedding, $embedding) AS score
        WHERE score >= $min_score
        ORDER BY score DESC
        LIMIT $top_k

        OPTIONAL MATCH (d:Document)-[:HAS_CHUNK]->(c)

        CALL {
            WITH c
            OPTIONAL MATCH (c)-[:MENTIONS]->(e:Entity)
            WITH e WHERE e IS NOT NULL
            OPTIONAL MATCH (e)-[r:REL]-(related:Entity)
            RETURN collect(DISTINCT {
                entity: e.name,
                type: e.type,
                relation: coalesce(r.type, ''),
                related_entity: coalesce(related.name, '')
            }) AS graph_context
        }

        RETURN d.id AS document_id,
               c.id AS chunk_id,
               c.name AS chunk_name,
               c.text AS chunk_text,
               score,
               graph_context
        ORDER BY score DESC
        """,
        {"embedding": q_emb, "top_k": top_k, "min_score": min_score},
    )

    if not results:
        print("No se encontraron chunks relevantes.")
        return []

    for row in results:
        print(f"\n=== {row['chunk_name']} | score: {row['score']:.3f} ===")
        print(row['chunk_text'][:500].replace("\n", " "))
        seen = set()
        for item in row['graph_context']:
            if not item['entity'] or item['entity'] in seen:
                continue
            seen.add(item['entity'])
            print(f"   [{item['type']}] {item['entity']}")
            if item['related_entity']:
                print(f"      \u2514\u2500\u2500 {item['relation']} \u2192 {item['related_entity']}")

    return results


graphrag_query("\u00bfQu\u00e9 frameworks y herramientas se usan para construir agentes?")


---
## 9. Limpieza

In [ ]:
# Cerrar conexión Neo4j
if _driver:
    _driver.close()
    print("Conexión Neo4j cerrada")